In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

In [38]:
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.compose import ColumnTransformer

In [39]:
df = pd.read_csv('/kaggle/input/datasets/sanjanbm/modified-titanic-data/train.csv')
df.sample(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
563,564,0,3,"Simmons, Mr. John",male,NaN,0,0,SOTON/OQ 392082,8.05,NaN,S
159,160,0,3,"Sage, Master. Thomas Henry",male,NaN,8,2,CA. 2343,69.55,NaN,S
573,574,1,3,"Kelly, Miss. Mary",female,NaN,0,0,14312,7.75,NaN,Q


In [40]:
df.drop(columns = ['PassengerId','Name','Ticket','Cabin'], inplace = True)
df.sample(4)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
269,1,1,female,35.00,0,0,135.6333,S
670,1,2,female,40.00,1,1,39.0000,S
40,0,3,female,40.00,1,0,9.4750,S
305,1,1,male,0.92,1,2,151.5500,S


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [42]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [43]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

X_train.shape, X_test.shape

((712, 7), (179, 7))

In [44]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [45]:
y_train.head()

331    0
733    0
382    0
704    0
813    0
Name: Survived, dtype: int64

## Apply column transformer

In [46]:
# Imputation transformer

tf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]), 
], remainder='passthrough')

tf1

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [47]:
# One hot encoding

tf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [1, 6])
], remainder='passthrough')

In [48]:
# scaling

tf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 10))
])

In [49]:
# Feature selection

tf4 = SelectKBest(score_func=chi2, k=5)

In [50]:
# train the model

tf5 = DecisionTreeClassifier()

## Create Pipeline

In [51]:
pipe = Pipeline([
    ('trf1', tf1),
    ('trf2', tf2),
    ('trf3', tf3),
    ('trf4', tf4),
    ('trf5', tf5),
])

### Pipeline Vs make_pipeline
Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

In [53]:
# Alternate Syntax
pipe = make_pipeline(tf1,tf2,tf3,tf4,tf5)

In [54]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selectkbest',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x787234e1d8a0>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

## Explore the Pipeline

In [55]:
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'selectkbest': SelectKBest(k=5, score_func=<function chi2 at 0x787234e1d8a0>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

In [56]:
# Display Pipeline

from sklearn import set_config
set_config(display='diagram')

In [57]:
y_pred = pipe.predict(X_test)

In [58]:
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0])

In [63]:
from sklearn.metrics import accuracy_score
print(f"{accuracy_score(y_test,y_pred)*100}%")

62.56983240223464.2f%


## Cross Validation using Pipeline

In [66]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.6391214419383433)

## GridSearch using Pipeline

In [72]:
pipe = Pipeline([
    ('imputer', tf1),
    ('encoder', tf2),
    ('scaler', tf3),
    ('feature_selection', tf4),
    ('model', DecisionTreeClassifier()) # Let's call it 'model'
])

# Then your params grid is much cleaner:
params = {
    'model__max_depth': [3, 5, 10],
    'feature_selection__k': [2, 5, 8]
}

In [73]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('imputer',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('encoder',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('scaler',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('feature_selection',
                                        SelectKBest(k=5,
                                                    score_func=<function chi2 at 0x787234e1d8a0>)),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={'feature_selection__k': [2, 5, 8],
                         'model__max_depth': [3, 5, 10]},
             scoring='accuracy')

In [74]:
grid.best_score_

np.float64(0.6391214419383433)

In [75]:
grid.best_params_

{'feature_selection__k': 2, 'model__max_depth': 3}

In [76]:
# export 
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))